# Core Imports

In [ ]:
# Generic Imports
import re
from collections import defaultdict
from ast import literal_eval

# Numeric imports
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# File I/O
from pathlib import Path
import csv, json

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import rdChemReactions
from rdkit.Chem.Draw import IPythonConsole

from polymerist.rdutils import set_rdkdraw_size
set_rdkdraw_size(300, 3/2)

# Inspecting monomer dataset

In [ ]:
from src.utils.dataIO import read_monomer_data


DATA_DIR = Path('src/monomer_data')

input_data_path = DATA_DIR / 'PolyID_master.csv'
# input_data_path = DATA_DIR / 'subsample_IPU.csv'

# load and parse monomer data file from disc
monomer_df = read_monomer_data([input_data_path])[0]
monomer_df.set_index(monomer_df.columns[0], inplace=True) # alternative to index_cols arg in read function
print(len(monomer_df))

# generate chemically-unique counterpart
monomer_df_unique = monomer_df.drop_duplicates('smiles_canonical')
print(len(monomer_df_unique))

## Sort by polymerization mechanism and #monomers, obtain respective counts and colors

In [ ]:
from polymerist.graphics import plotutils

keys = ['mechanism']#, 'num_monomers']

grouper = monomer_df.groupby(keys, as_index=False) # group by mechanism name, preserving original indices
counts : pd.DataFrame = grouper.size().sort_values(by='size')
frames : dict[str, pd.DataFrame] = {
    mech : grouper.get_group(mech)
        for mech in grouper.groups
}

In [ ]:
dfs : dict[str, pd.DataFrame] = {
    'total'  : monomer_df,
    'unique' : monomer_df_unique,
}
prop : str = 'mechanism'

# generate colorbar with unique color for each mechanism
cdict, carr = plotutils.label_discrete_cmap(
    cmap=plt.get_cmap('tab10'),
    color_names=monomer_df[prop].unique(),
    hues_per_color=1
)
# plt.imshow(carr)

# plot distributions of property across variants of dataframe
fig, ax = plotutils.presize_subplots(1, 1, scale=10, elongation=2/3)
alpha_dropoff = 1.0 / len(dfs)

title_counts : list[str] = []
for label, dataframe in dfs.items(): 
    title_counts.append(f'{len(dataframe)} {label}')
    
    prop_counts : pd.Series = dataframe[prop].value_counts()
    bar_labels, bar_heights = prop_counts.index, prop_counts.values
    bar_colors = [cdict[label][:-1] + (alpha_dropoff,) for label in bar_labels] # split transparency uniformly amongst dataframes
    
    bars = ax.bar(bar_labels, bar_heights, color=bar_colors)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), bar.get_height(), ha='center', va='bottom')
        
count_str = ', '.join(title_counts)        
ax.set_title(f'Number of SMILES by {prop} ({count_str})')
# fig.savefig(f'monomer_data_raw/monomers_by_{prop}.png', bbox_inches='tight')

# Analyzing distribution of molecule sizes

In [ ]:
from pathlib import Path
from rdkit import Chem

from src.utils.dataIO import read_monomer_data


DATA_DIR : Path = Path('src/monomer_data')
DATA_PATHNAME : str = 'PolyID_master_unique.csv'

# load and parse monomer data file from disc
input_data_path = DATA_DIR / DATA_PATHNAME
monomer_df = read_monomer_data([input_data_path])[0]
monomer_df.set_index(monomer_df.columns[0], inplace=True) # alternative to index_cols arg in read function

# grouping chemistries by labelled polymerization mechanism
grouper = monomer_df.groupby('mechanism_labelled', as_index=False)
sample = grouper.sample(n=1)
print(grouper.groups.keys())

In [ ]:
from typing import Any
from rich.progress import track
import pandas as pd

from rdkit.Chem.Descriptors import ExactMolWt, MolWt
from rdkit.Chem.rdmolops import AROMATICITY_MDL, SANITIZE_ALL

from polymerist.smileslib.cleanup import canonical_SMILES_from_mol
from polymerist.rdutils.sanitization import sanitize_mol
from polymerist.rdutils.substructures import num_automorphisms


molecule_records : list[dict[str, Any]] = []
for i, row in track(monomer_df.iterrows(), description='Characterizing unique molecules...', total=len(monomer_df)):
    monomer_mol_combined = Chem.MolFromSmiles(row.smiles_explicit, sanitize=False)
    sanitize_mol(monomer_mol_combined, sanitize_ops=SANITIZE_ALL, aromaticity_model=AROMATICITY_MDL, in_place=True)
    monomer_mols = Chem.GetMolFrags(monomer_mol_combined, sanitizeFrags=False, asMols=True)

    # get number of monomers per chemical entry
    monomer_df.at[i, 'num_monomers'] = len(monomer_mols)
    
    # strip out individual molecules
    for mol in monomer_mols:
        molecule_records.append(
            {
                'smiles_canonical' : canonical_SMILES_from_mol(mol),
                'molecular_weight' : ExactMolWt(mol),
                'n_atoms' : mol.GetNumAtoms(),
                'n_automorphisms' : num_automorphisms(mol, maxMatches=1_000),
            }
        )
monomer_df.num_monomers = monomer_df.num_monomers.astype(int)

# compile molecule-specific dataframe
molecule_df = pd.DataFrame.from_records(molecule_records)
molecule_df.drop_duplicates(subset=['smiles_canonical'], inplace=True)
molecule_df.sort_values(by=['molecular_weight', 'n_atoms'], ascending=False, inplace=True)

display(molecule_df)

In [ ]:
molecule_df.hist('n_automorphisms', grid=False, bins=40)

In [ ]:

import matplotlib.pyplot as plt
from math import floor, sqrt


molsize_imgs_path = Path('molecule_sizes')
molsize_imgs_path.mkdir(exist_ok=True)

plt_cfg = {
    'molecular_weight' : ('Molecular weight', 1_000),
    'n_atoms' : ('Number of atoms', 150),
}

quantiles : list[tuple[float, str]] = [
    (0.90, 'b'),
    (0.95, 'm'),
]
                                       
scale : float = 6
aspect : float = 2.7
fig, axes = plt.subplots(1, len(plt_cfg), figsize=(scale*aspect, scale))
n_bins = floor(sqrt(len(molecule_df)))

for ax, (df_field, (title, cutoff)) in zip(axes, plt_cfg.items()):
    data = molecule_df[df_field]
    n_outliers = len(data[data >= cutoff])
    
    ax.hist(molecule_df[df_field], bins=n_bins)
    for quantile, color in quantiles:
        quantile_cutoff = data.quantile(quantile)
        ax.axvline(quantile_cutoff, linestyle='-.', color=color, label=f'{quantile*100}% quantile ({round(quantile_cutoff)})')
    ax.axvline(cutoff, linestyle='--', color='r', label=f'Arbitrary threshold ({cutoff})')
        
    ax.set_title(f'{title} ({n_outliers} above cutoff of {cutoff})')
    ax.set_xlabel(df_field)
    ax.set_ylabel('Counts')
    
    ax.legend()

fig.savefig(molsize_imgs_path / 'mol_size_distrib.png', bbox_inches='tight')
molecule_df.to_csv(molsize_imgs_path / 'PolyID_master_unique_molecules.csv')

In [ ]:
from rdkit.Chem.Draw import MolsToGridImage, MolToImageFile

n = 1000
heaviest_mols = []
for i, row in molecule_df.sort_values(by='molecular_weight', ascending=False).head(6).iterrows():
    molecule = Chem.MolFromSmiles(row.smiles_canonical, sanitize=False)
    sanitize_mol(molecule, sanitize_ops=SANITIZE_ALL, aromaticity_model=AROMATICITY_MDL, in_place=True)
    molecule = Chem.AddHs(molecule)
    display(molecule)
    print(i, row.molecular_weight, row.n_atoms)
    heaviest_mols.append(molecule)
    
    MolToImageFile(molecule, filename=str(molsize_imgs_path / f'mol_{i}.png'), size=(n, n), kekulize=False)

## Inspecting vinyl R-groups

In [ ]:
from typing import Generator, Iterable, Optional, Sequence

from polymerist.rdutils.labeling.molwise import (
    atom_ids_by_map_nums,
    assign_ordered_atom_map_nums,
    has_fully_mapped_atoms,
    clear_atom_map_nums,
    clear_atom_isotopes,
)

def extract_vinyl_R_groups(
        molecule : Chem.Mol, 
        mapped_query_mol : Chem.Mol, 
        r_group_bond_atom_nums : Iterable[tuple[int, int]], 
        dummy_labels : Optional[Sequence[tuple[int, int]]]=None,
    ) -> Generator[Chem.Mol, None, None]:
    '''yields molecule fragments resulting from removal of each double-bonded carbon found in the molecule'''
    # sanitize dummy labels list
    if dummy_labels is None:
        dummy_labels = []
    else:
        assert len(dummy_labels) == len(r_group_bond_atom_nums)
    
    # query for vinyls
    mapped_molecule = assign_ordered_atom_map_nums(molecule)
    assert has_fully_mapped_atoms(mapped_molecule)
    assert has_fully_mapped_atoms(mapped_query_mol)
    
    for match_indices in mapped_molecule.GetSubstructMatches(mapped_query_mol, useChirality=False):
        modmol = Chem.RWMol(mapped_molecule) # create new editable copy of the molecule for each vinyl bond
        # generate correspondence between query map numbers and atom ids in queried molecule
        query_map_to_match_id : dict[int, int] = { # map from query map numbers to atom indices in the match
            atom.GetAtomMapNum() : match_id # relies on the fact that match atoms are returned in the same order as they appear in the query Mol
                for (atom, match_id) in zip(mapped_query_mol.GetAtoms(), match_indices)
        }
        
        # identify indices of objects to remove
        bond_ids_to_cut : list[int] = sorted((
                modmol.GetBondBetweenAtoms(
                    query_map_to_match_id[bond_start],
                    query_map_to_match_id[bond_end],
                ).GetIdx()
                    for (bond_start, bond_end) in r_group_bond_atom_nums
            ),
            reverse=True, # sort in descending order so each deletion doesn't affect those that follow it
        )
        
        # remove atom and bond targets and yield appropriate fragments
        frag_mol = Chem.FragmentOnBonds(modmol, bondIndices=bond_ids_to_cut, addDummies=True, dummyLabels=dummy_labels)
        for fragment in Chem.GetMolFrags(frag_mol, asMols=True):
            if fragment.GetSubstructMatch(mapped_query_mol): # don't return the central vinyl carbon fragments
                continue
            
            yield fragment

In [ ]:
# define vinyl group query and associated atom map numbers
# CC_DOUBLE_BOND_QUERY = Chem.MolFromSmarts('[C:1](-[*:3])(-[*:4])=[C:2](-[*:5])-[*:6]')
# CC_DOUBLE_BOND_ATOM_MAP_NUMS : tuple[int] = (1, 2) # map numbers on the vinyl carbons
# R_GROUP_MAP_NUMS : tuple[int] = (3, 4, 5, 6)  # map numbers of R-group bridgehead atoms

VINYL_QUERY = Chem.MolFromSmarts('[C:1](-[*:3])(-[H:4])=[C:2](-[H:5])-[H:6]')
VINYL_ATOM_MAP_NUMS : tuple[int] = (1, 2) # map numbers on the vinyl carbons
R_GROUP_MAP_NUMS : tuple[int] = (3,)  # map numbers of R-group bridgehead atoms

R_GROUP_BOND_MAP_NUMS : list[tuple[int, int]] = [  # select R-group-to-vinyl bonds for severance
    (vinyl_map_num, r_group_map_num)
        for vinyl_map_num in VINYL_ATOM_MAP_NUMS
            for r_group_map_num in R_GROUP_MAP_NUMS
                if VINYL_QUERY.GetBondBetweenAtoms(*atom_ids_by_map_nums(VINYL_QUERY, vinyl_map_num, r_group_map_num)) is not None
]

display(VINYL_QUERY)
print(R_GROUP_BOND_MAP_NUMS)

### Extract out unique R-group fragments by query

In [ ]:
from rdkit import RDLogger                                                                                                                                                               
from rich.progress import track


unique_R_groups : set[Chem.Mol]  = set()
unique_R_group_SMILES : set[str] = set()

subframe = monomer_df_unique.groupby('mechanism').get_group('vinyl')
num_mols = len(subframe)

not_wholly_vinylic : list[tuple[int, str]] = [] # catalogue all monomer(s) for which not ALL parts are vinyls
RDLogger.DisableLog('rdApp.warning') # temporarily suppress unmapped hydrogen warnings
for i, row in track(subframe.iterrows(), description='Extracting unique vinyl R-groups...', total=num_mols):
    combo_mol = Chem.MolFromSmiles(row.smiles_explicit, sanitize=False)

    all_parts_vinylic : bool = True
    for indiv_mol in Chem.GetMolFrags(combo_mol, asMols=True):
        if not indiv_mol.HasSubstructMatch(VINYL_QUERY):
            all_parts_vinylic = False
            continue
        
        for frag in extract_vinyl_R_groups(
            indiv_mol, 
            mapped_query_mol=VINYL_QUERY, 
            r_group_bond_atom_nums=R_GROUP_BOND_MAP_NUMS, 
            dummy_labels=[(j, j) for (i, j) in R_GROUP_BOND_MAP_NUMS]
        ):
            unmapped_frag = clear_atom_map_nums(clear_atom_isotopes(frag))
            canon_smi = Chem.CanonSmiles(Chem.MolToSmiles(unmapped_frag))
            if canon_smi not in unique_R_group_SMILES:
                unique_R_group_SMILES.add(canon_smi)
                unique_R_groups.add(unmapped_frag)
    
    if not all_parts_vinylic:
        not_wholly_vinylic.append((i, row.smiles_explicit, row.smiles_canonical))
RDLogger.EnableLog('rdApp.warning') # re-enable warnings for other code

In [ ]:
fake_vinyls = pd.DataFrame.from_records( # convert records to dataframe for ease of indexing
    not_wholly_vinylic, columns=[
        'vinyl_index',
         'smiles_explicit',
         'smiles_canonical',
    ],
    index='vinyl_index'
)       

fake_vinyl_canon_smi = set(fake_vinyls['smiles_canonical']) # cache set to avoid lookup for each function call
fake_vinyl_idxs = monomer_df_unique['smiles_canonical'].map(lambda s : s in fake_vinyl_canon_smi)

monomer_df_unique_real_vinyls = monomer_df_unique[~fake_vinyl_idxs]
monomer_df_unique_real_vinyls_fmt = monomer_df_unique[~fake_vinyl_idxs]

In [ ]:
# save results to file
outdir = Path('monomer_data_fmt')
outdir.mkdir(exist_ok=True)

vinyl_R_groups_path : Path = (outdir / 'unique_vinyl_R_groups.smi')
with vinyl_R_groups_path.open('w') as file:
    for smiles in sorted(unique_R_group_SMILES):
        file.write(f'{smiles}\n')

monomer_df_unique_real_vinyls_fmt.to_csv(outdir / 'PolyID_master_unique_true_vinyls.csv')

## Identifying monomers with mislabelled reactions

In [ ]:
import json
from polymerist.rdutils.reactions.reactions import AnnotatedReaction, BadNumberReactants

# rxn_smarts_file = Path('src/reactions/rxn_smarts.json')
rxn_smarts_file = Path('src/reactions/rxns_polyID.json')
with rxn_smarts_file.open('r') as file:
    rxn_smarts_registry = json.load(file)
    rxns = {
        rxn_name : AnnotatedReaction.from_smarts(rxn_smarts)
            for rxn_name, rxn_smarts in rxn_smarts_registry.items()
    }

In [ ]:
def load_rdmol(smiles : str) -> Chem.Mol:
    reactant_mol = Chem.MolFromSmiles(smiles, sanitize=False) # CRITICAL that sanitize=False to avoid stripping
    Chem.SanitizeMol(reactant_mol) # single, unified mol containing individual reactant as disconnected components
    
    return reactant_mol

def get_compatible_rxn_names(smiles : str) -> set[str]:
    reactant_mol = load_rdmol(smiles)
    reactants = Chem.GetMolFrags(reactant_mol, asMols=True)
    
    compat_rxn_names = set()
    for rxnname, rxn in rxns.items():
        try:
            if rxn.valid_reactant_ordering(reactants) is not None:
                compat_rxn_names.add(rxnname)
        except BadNumberReactants:
            pass
            
    return compat_rxn_names

In [ ]:
monomer_df_unique['compatible_rxns'] = monomer_df_unique['smiles_explicit'].map(get_compatible_rxn_names)
have_matches = monomer_df_unique['compatible_rxns'] != set()

labelled_accurately = monomer_df_unique.apply(lambda row: row['mechanism'] in row['compatible_rxns'], axis=1)
print(have_matches.sum(), labelled_accurately.sum(), have_matches.sum() - labelled_accurately.sum())

In [ ]:
monomer_df_unique[have_matches & ~labelled_accurately][['smiles_canonical', 'mechanism', 'compatible_rxns']].to_csv('mislabelled_rxns.csv')

In [ ]:
from polymerist.rdutils import disable_kekulized_drawing
disable_kekulized_drawing()

n_shown : int = 10
subframe = monomer_df_unique[~have_matches]
# subframe = monomer_df_unique[have_matches & ~labelled_accurately]

for i, row in subframe.head(n_shown).iterrows():
    unimol = load_rdmol(row.smiles_explicit)
    mols_list = Chem.GetMolFrags(unimol, asMols=True)
    
    display(unimol)
    print(len(mols_list), row.mechanism, row.compatible_rxns, row.smiles_canonical)